# NBS site query — flood E2E (Porto Alegre)

Bairro-level flood screening + **250 m grid** intra-bairro differentiation.

Follows [`recommended-datasets.md`](../docs/recommended-datasets.md) and [`flood_nbs_dataset_lens.md`](../docs/flood_nbs_dataset_lens.md).

| Section | Unit | Purpose |
|---------|------|---------|
| **Bairro** | Bairro polygon | Priority + mechanism + flood NBS |
| **Grid** | 250 m cell | Per-cell mechanism flags + dominant NBS |

**Default site:** Humaitá — high flood risk, adjacent to Rio Gravataí

**CLI:** `run_e2e.py --hazard flood`

**Setup:** run with cwd = `scripts/`; use nbs_e2e venv or floods `.venv`.


## Setup — paths and imports


In [ ]:
import json
import os
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

# Jupyter kernels often omit Homebrew from PATH on macOS.
for _brew_bin in (Path("/opt/homebrew/bin"), Path("/usr/local/bin")):
    if _brew_bin.is_dir():
        _p = str(_brew_bin)
        if _p not in os.environ.get("PATH", "").split(":"):
            os.environ["PATH"] = f"{_p}:{os.environ.get('PATH', '')}"

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
from rasterio.transform import array_bounds
from shapely.geometry import mapping

NOTEBOOK_DIR = Path.cwd()
NBS_E2E_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "scripts" else NOTEBOOK_DIR
SCRIPTS_DIR = NBS_E2E_ROOT / "scripts"
if not (SCRIPTS_DIR / "catalog_layers.py").exists():
    raise FileNotFoundError(
        "Run this notebook with cwd = transformation/nbs_screening/scripts "
        f"(expected catalog_layers.py under {SCRIPTS_DIR})"
    )
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

for mod in ("catalog_layers", "nbs_rules", "grid_screening"):
    sys.modules.pop(mod, None)
import catalog_layers as _catalog_layers
import nbs_rules as _nbs_rules

OUT_DIR = NBS_E2E_ROOT / "output"
IN_DIR = OUT_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

import grid_screening as _grid_screening

HAZARD = "flood"
CATALOG_COGS = _catalog_layers.get_catalog_cogs(HAZARD)
LOCAL_SCREENING_RASTERS = _catalog_layers.get_local_rasters(HAZARD)
barrio_flood_context = _catalog_layers.barrio_flood_context
query_all_layers = _catalog_layers.query_all_layers
query_layers = _catalog_layers.query_layers
recommend_all = _nbs_rules.recommend_all
NBS_TYPES = _nbs_rules.NBS_TYPES
screen_bairro_grid = _grid_screening.screen_bairro_grid
screen_poa_flood_mechanism_grid = _grid_screening.screen_poa_flood_mechanism_grid
cells_to_geodataframe = _grid_screening.cells_to_geodataframe
result_to_geojson = _grid_screening.result_to_geojson
result_to_report_dict = _grid_screening.result_to_report_dict
export_flood_mechanism_geotiff = _grid_screening.export_flood_mechanism_geotiff
export_poa_flood_mechanism_layers = _grid_screening.export_poa_flood_mechanism_layers
MECHANISM_RASTER_NODATA = _grid_screening.MECHANISM_RASTER_NODATA

FLOOD_MECHANISM_TYPE_CODES = _nbs_rules.FLOOD_MECHANISM_TYPE_CODES
FLOOD_MECHANISM_CATALOG_DOCS = _nbs_rules.FLOOD_MECHANISM_CATALOG_DOCS
MECHANISM_MIXED_COLOR = _nbs_rules.MECHANISM_MIXED_COLOR
classify_dominant_flood_mechanism = _nbs_rules.classify_dominant_flood_mechanism

MECHANISM_COLORS = {
    "none": "#e0e0e0",
    "riverine": "#2166ac",
    "pluvial": "#fdae61",
    "low_lying": "#abd9e9",
    "drainage_constrained": "#7b3294",
    "mixed": MECHANISM_MIXED_COLOR,
}

GRID_LAYER_IDS = {"app_hev_250m", "sample_grid_1km"}

required_catalog_layers = {
    "flood_hazard", "exposure", "vulnerability", "flood_risk",
    "merit_hand", "merit_upa", "ghsl_built_up",
}
missing = required_catalog_layers - CATALOG_COGS.keys()
if missing:
    raise KeyError(f"CATALOG_COGS missing expected layers: {sorted(missing)}")

SITE_NAME = "Humaitá"  # change to test another bairro
print("NBS E2E root:", NBS_E2E_ROOT)
print("Hazard:", HAZARD)
print("Site:", SITE_NAME)
print("Catalog layers:", list(CATALOG_COGS.keys()))
print("Local rasters:", list(LOCAL_SCREENING_RASTERS.keys()))


# Bairro-level flood screening

## Step 0 — Priority screening (where to start)

Load the bairro polygon and flood **risk / exposure / vulnerability** from OEF outputs.

Priority ≠ suitability — this only tells us screening is warranted here.


In [ ]:
ctx_row = barrio_flood_context(SITE_NAME)
site_geom = ctx_row.pop("geometry")

site_gdf = gpd.GeoDataFrame([{"name": SITE_NAME, "geometry": site_geom}], crs="EPSG:4326")
site_path = IN_DIR / f"site_{SITE_NAME.lower().replace(' ', '_')}.geojson"
site_gdf.to_file(site_path, driver="GeoJSON")

step0 = pd.Series({
    "bairro": SITE_NAME,
    "hazard_mean": ctx_row["hazard_mean"],
    "risk_mean": ctx_row["risk_mean"],
    "exposure_score": ctx_row["exposure_score"],
    "vulnerability_score": ctx_row["vulnerability_score"],
})
display(step0.to_frame("value"))
print(f"Site saved → {site_path}")
print("Bounds (lon/lat):", site_geom.bounds)


## Step 1a — Query diagnostic layers

**Target:** mask app/catalog COGs (S3) to the site polygon → zonal mean / median / p90.

**Screening grid:** use the same app-accessed COGs for flood H/E/V/R, MERIT HAND/UPA and GHSL built-up. Local ~250 m rasters are kept only as fallback if the catalog COGs are unavailable.


In [ ]:
layers = query_all_layers(site_geom)

rows = []
for layer in layers:
    row = {
        "layer_id": layer.layer_id,
        "status": layer.status,
        "note": layer.note[:80] if layer.note else "",
    }
    for k, v in layer.stats.items():
        row[k] = round(v, 4) if isinstance(v, float) else v
    rows.append(row)

layers_df = pd.DataFrame(rows)
display(layers_df)

### Inspect app/catalog screening metrics (inside bairro)

These proxies drive mechanism inference using the same app-accessed COGs where available; missing H/E/V/R metrics fall back to the local 250 m grid.


In [ ]:
grid_layer = next(l for l in layers if l.layer_id == "app_hev_250m")
water_layer = next(l for l in layers if l.layer_id == "osm_waterways")

grid_stats = grid_layer.stats
water_stats = {**water_layer.stats, "_note": water_layer.note}

print("Screening pixels in site:", grid_stats.get("n_cells"))
print("Source:", grid_layer.note)
display(pd.Series({k: v for k, v in grid_stats.items() if k != "n_cells"}).to_frame("mean"))
print("Waterways:", water_stats)


## Step 1b — Infer flood mechanism

Before choosing an NBS type: *why* might water collect here?


In [ ]:
ctx = {"bairro": SITE_NAME, **ctx_row}
for layer in layers:
    if layer.status == "ok" and layer.layer_id in CATALOG_COGS:
        ctx[f"{layer.layer_id}_mean"] = layer.stats.get("mean")
    elif layer.status == "ok" and layer.layer_id == "cougar_hazard_250m":
        ctx["local_flood_hazard_mean"] = layer.stats.get("mean")
    elif layer.status == "ok" and layer.layer_id == "cougar_risk_250m":
        ctx["local_flood_risk_mean"] = layer.stats.get("mean")

mechanism, recommendations = recommend_all(ctx, grid_stats, water_stats)

mech_df = pd.DataFrame({
    "signal": ["riverine", "pluvial", "low_lying", "drainage_constrained (gap)"],
    "value": [mechanism.riverine, mechanism.pluvial, mechanism.low_lying, mechanism.drainage_constrained],
})
display(mech_df)
print("\nRationale:")
for line in mechanism.rationale:
    print(" •", line)

## Step 2 — NBS typology screening

Simple rule scores (0–1) per NBS type. **≥ 0.55 = plausible** for early ideation only — not engineering design.


In [ ]:
recs_df = pd.DataFrame([asdict(r) for r in recommendations])
recs_df["gaps"] = recs_df["gaps"].apply(lambda g: "; ".join(g) if g else "")
display(recs_df[["nbs_type", "score", "rationale", "gaps"]])

print("\nTop 3 for ideation:")
for _, row in recs_df.head(3).iterrows():
    print(f"  [{row['score']:.2f}] {row['nbs_type']}")

## Step 6 — Gaps discovered

Catalog errors + rule-level data needs before site design.


In [ ]:
gaps = []
for layer in layers:
    if layer.status == "error":
        gaps.append(f"[{layer.layer_id}] {layer.note}")
for rec in recommendations:
    gaps.extend(rec.gaps)
gaps = sorted(set(gaps))

for i, g in enumerate(gaps, 1):
    print(f"{i}. {g}")

## Save report (JSON)


In [ ]:
report = {
    "exercise": "nbs_site_query_flood_e2e",
    "hazard": "flood",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "site": {"name": SITE_NAME, "geojson": str(site_path)},
    "step_0_priority": ctx,
    "step_1_layers": [
        {
            "layer_id": l.layer_id,
            "source": l.source,
            "status": l.status,
            "stats": l.stats,
            "note": l.note,
        }
        for l in layers
    ],
    "step_1_mechanism": {
        "riverine": mechanism.riverine,
        "pluvial": mechanism.pluvial,
        "low_lying": mechanism.low_lying,
        "rationale": mechanism.rationale,
    },
    "step_2_nbs_recommendations": [asdict(r) for r in recommendations],
    "gaps_discovered": gaps,
}

out_path = OUT_DIR / f"nbs_site_query_flood_{SITE_NAME.lower().replace(' ', '_')}.json"
out_path.write_text(json.dumps(report, indent=2, ensure_ascii=False))
print(f"Report written → {out_path}")

# Grid screening — flood mechanism type layer (ON-5990)

**Primary unit = 250 m grid cell**, not bairro average. The bairro polygon (`SITE_NAME`) filters which cells to screen for this exercise.

Each cell gets:

| Field | Description |
|-------|-------------|
| `flood_mechanism_type` | **Dominant** mechanism: `riverine` · `pluvial` · `low_lying` · `drainage_constrained` · `mixed` · `none` |
| `flood_mechanism_code` | Integer code for catalog raster (0–5) |
| Boolean flags | `riverine`, `pluvial`, `low_lying` (can overlap; type picks dominant by strength) |
| `dominant_nbs` | Top-scoring NBS typology for that cell |

Logic lives in `nbs_rules.classify_dominant_flood_mechanism()` — globally applicable proxies only.

## one neighborhood

In [ ]:
# ~20–60 s for a typical bairro (Humaitá ≈ 56 cells). Progress prints every ~10%.
print(f"Screening 250 m grid cells for {SITE_NAME}...", flush=True)
grid_result = screen_bairro_grid(SITE_NAME, hazard="flood", sample_catalog=True)
grid_gdf = cells_to_geodataframe(grid_result)

print(f"Cells screened: {grid_result.n_cells} @ ~{grid_result.cell_size_m:.0f} m")
print("\nBairro mechanism rollup (derived from cells):")
display(pd.Series(grid_result.mechanism_summary).to_frame("value"))

print("\nDominant flood_mechanism_type counts:")
if "dominant_mechanism_type_counts" in grid_result.mechanism_summary:
    display(
        pd.Series(grid_result.mechanism_summary["dominant_mechanism_type_counts"])
        .sort_values(ascending=False)
        .to_frame("cells")
    )

display(
    grid_gdf[
        [
            "cell_id",
            "flood_mechanism_type",
            "flood_mechanism_code",
            "riverine",
            "pluvial",
            "low_lying",
            "dist_river_m",
            "flood_score",
            "dominant_nbs",
        ]
    ].sort_values("dist_river_m")
)

### Mechanism type codes (catalog)

See `FLOOD_MECHANISM_CATALOG_DOCS` in setup / `nbs_rules.py`.

### Map — dominant `flood_mechanism_type` per cell

| Color | Type |
|-------|------|
| Blue | riverine |
| Orange | pluvial |
| Light blue | low_lying |
| Purple | drainage_constrained |
| Dark magenta | mixed (multiple competing mechanisms) |
| Light gray | none |

In [ ]:
# MECHANISM_COLORS defined in setup cell
grid_gdf = grid_gdf.copy()
grid_gdf["color"] = grid_gdf["flood_mechanism_type"].map(MECHANISM_COLORS).fillna("#cccccc")

fig, ax = plt.subplots(figsize=(9, 8))
site_gdf.boundary.plot(ax=ax, color="black", linewidth=1.5, label=SITE_NAME)
grid_gdf.plot(ax=ax, color=grid_gdf["color"], edgecolor="white", linewidth=0.3, alpha=0.9)
ax.set_title(f"Dominant flood mechanism type — {SITE_NAME} (250 m)")
ax.set_xlabel("lon")
ax.set_ylabel("lat")

from matplotlib.patches import Patch
legend_handles = [
    Patch(facecolor=c, edgecolor="white", label=t) for t, c in MECHANISM_COLORS.items()
]
ax.legend(handles=legend_handles, loc="lower left", fontsize=8, title="flood_mechanism_type")
plt.tight_layout()
plt.show()

### Save grid outputs

In [ ]:
grid_slug = SITE_NAME.lower().replace(" ", "_")
grid_geojson_path = OUT_DIR / f"nbs_grid_flood_{grid_slug}.geojson"
grid_json_path = OUT_DIR / f"nbs_grid_flood_{grid_slug}.json"
grid_tif_path = OUT_DIR / f"flood_mechanism_type_{grid_slug}_250m.tif"

geojson_payload = result_to_geojson(grid_result)
geojson_payload["properties"]["layer"] = "poa_flood_mechanism_type"
geojson_payload["properties"]["mechanism_type_codes"] = FLOOD_MECHANISM_TYPE_CODES
grid_geojson_path.write_text(json.dumps(geojson_payload, indent=2, ensure_ascii=False))
grid_json_path.write_text(json.dumps(result_to_report_dict(grid_result), indent=2, ensure_ascii=False))
export_flood_mechanism_geotiff(grid_result, grid_tif_path)

print(f"GeoJSON → {grid_geojson_path}")
print(f"Report  → {grid_json_path}")
print(f"GeoTIFF → {grid_tif_path}")
print("\n", FLOOD_MECHANISM_CATALOG_DOCS)

## full POA catalog layer

Set `BUILD_POA_LAYER = True` to build the city-wide mechanism raster.

**Screening strategy (fast path):**
- Classifies **~15.4k hazard-valid** pixels directly (catalog layers preloaded once into memory).
- **~3.9k gap** pixels outside hazard coverage are **not** screened cell-by-cell — `export_poa_flood_mechanism_layers()` fills them with **IDW** from neighbor strengths.
- Expect **~5–15 min** after preload (not hours). If a prior run is still going at ~16 s/cell, **interrupt the kernel** and re-run this cell.

**Methodology, assumptions, and limitations:** [`docs/poa_mechanism_type_layer.md`](../docs/poa_mechanism_type_layer.md)

**Outputs (cascade, like hazard base + IDW):**

| File | Role |
|---|---|
| `flood_mechanism_type_poa_250m_observed.tif` | Direct screening on hazard-valid pixels (~15.4k) |
| `flood_mechanism_type_poa_250m.tif` | **Filled** layer for tiles (observed + IDW strengths in gaps) |
| `flood_mechanism_is_interpolated_poa_250m.tif` | Mask: 1 = IDW-filled pixel |
| `flood_mechanism_type_poa_250m.geojson` | Hazard-valid screened cells (+ `hazard_valid`, `is_interpolated` after export) |

Run the **POA map** cell (uses filled `.tif`), then **`PUBLISH_POA_COG_TILES = True`** to build COG + XYZ tiles and upload **COG, tiles, and GeoJSON** to S3 (`UPLOAD_POA_TO_S3`).

In [ ]:
BUILD_POA_LAYER = True  # flip to True for full-city catalog raster

if BUILD_POA_LAYER:
    poa_result = screen_poa_flood_mechanism_grid(sample_catalog=True, include_nbs=False)
    poa_paths = export_poa_flood_mechanism_layers(poa_result, OUT_DIR)
    poa_tif = poa_paths["filled"]
    poa_observed_tif = poa_paths["observed"]
    poa_interp_tif = poa_paths["is_interpolated"]
    poa_geojson = OUT_DIR / "flood_mechanism_type_poa_250m.geojson"
    poa_payload = result_to_geojson(poa_result)
    poa_payload["properties"]["layer"] = "poa_flood_mechanism_type"
    poa_payload["properties"]["mechanism_type_codes"] = FLOOD_MECHANISM_TYPE_CODES
    poa_payload["properties"]["raster_products"] = {
        "observed": poa_observed_tif.name,
        "filled": poa_tif.name,
        "is_interpolated": poa_interp_tif.name,
    }
    poa_payload["properties"]["mechanism_summary"] = {
        **poa_result.mechanism_summary,
        "hazard_valid_cell_count": sum(1 for c in poa_result.cells if c.hazard_valid),
        "interpolated_cell_count": sum(1 for c in poa_result.cells if c.is_interpolated),
    }
    poa_geojson.write_text(json.dumps(poa_payload, ensure_ascii=False))
    print(f"POA cells screened: {poa_result.n_cells}")
    print(f"POA observed TIF → {poa_observed_tif}")
    print(f"POA filled TIF   → {poa_tif}")
    print(f"POA interp mask  → {poa_interp_tif}")
    print(f"POA GeoJSON      → {poa_geojson}")
    display(pd.Series(poa_result.mechanism_summary.get("dominant_mechanism_type_counts", {})).to_frame("cells"))
else:
    print("Skipping full POA build (BUILD_POA_LAYER=False). Bairro exports above are sufficient for review.")

### Diagnose POA `none` mechanism cells

Reads the saved POA export (`output/flood_mechanism_type_poa_250m.geojson`) — no need to re-run `BUILD_POA_LAYER`. If `poa_result` is still in memory, per-cell strengths and rationale are shown too.

Summarizes **all POA** screened cells (~5k); per-cell detail is filtered to the current `SITE_NAME` bairro.

`flood_hazard` and `flood_mechanism_type` are independent — hazard can be present while every mechanism strength stays below `min_strength = 0.2`.


In [ ]:
MIN_STRENGTH = 0.2
POA_GEOJSON = OUT_DIR / "flood_mechanism_type_poa_250m.geojson"
POA_TIF = OUT_DIR / "flood_mechanism_type_poa_250m.tif"

poa_gdf = None
poa_mechanism_summary = None
poa_cells_by_id = {}
poa_source = None

if "poa_result" in globals():
    poa_gdf = cells_to_geodataframe(poa_result)
    poa_mechanism_summary = poa_result.mechanism_summary
    poa_cells_by_id = {c.cell_id: c for c in poa_result.cells}
    poa_source = "in-memory poa_result"
elif POA_GEOJSON.exists():
    poa_payload = json.loads(POA_GEOJSON.read_text())
    poa_gdf = gpd.read_file(POA_GEOJSON)
    poa_mechanism_summary = poa_payload.get("properties", {}).get("mechanism_summary", {})
    poa_source = POA_GEOJSON
elif POA_TIF.exists():
    print(
        f"GeoJSON not found at {POA_GEOJSON}. "
        f"Re-run BUILD_POA_LAYER once to export GeoJSON (TIF alone lacks per-cell attributes)."
    )
else:
    print(
        f"POA export not found. Run BUILD_POA_LAYER once, or place exports under {OUT_DIR}."
    )

if poa_gdf is not None:
    n_poa = len(poa_gdf)
    none_gdf = poa_gdf[poa_gdf["flood_mechanism_type"] == "none"]
    n_none = len(none_gdf)

    print(f"POA source: {poa_source}")
    print(
        f"POA mechanism_type='none': {n_none} / {n_poa} cells "
        f"({100 * n_none / max(n_poa, 1):.1f}%)"
    )

    type_counts = poa_mechanism_summary.get("dominant_mechanism_type_counts")
    if not type_counts and "flood_mechanism_type" in poa_gdf.columns:
        type_counts = poa_gdf["flood_mechanism_type"].value_counts().to_dict()
    display(
        pd.Series(type_counts or {})
        .sort_values(ascending=False)
        .to_frame("cells")
    )

    if n_none == 0:
        print("No 'none' cells in POA layer.")
    else:
        other_gdf = poa_gdf[poa_gdf["flood_mechanism_type"] != "none"]

        compare_cols = [
            "flood_score",
            "risk_score",
            "dist_river_m",
            "riverine",
            "pluvial",
            "low_lying",
        ]
        summary = []
        for label, subset in [("none", none_gdf), ("classified", other_gdf)]:
            row = {"group": label, "n_cells": len(subset)}
            for col in compare_cols:
                if col in subset.columns:
                    if subset[col].dtype == bool:
                        row[f"{col}_pct_true"] = 100 * subset[col].mean()
                    else:
                        row[f"{col}_mean"] = subset[col].mean()
                    row[f"{col}_missing"] = int(subset[col].isna().sum())
            summary.append(row)

        print("\n--- POA-wide: none vs classified ---")
        display(pd.DataFrame(summary).set_index("group"))

        if "site_geom" in globals():
            site_none_gdf = none_gdf[none_gdf.intersects(site_geom)]
            print(
                f"\n--- {SITE_NAME} bairro: {len(site_none_gdf)} none cells "
                f"(of {n_none} POA-wide) ---"
            )
        else:
            site_none_gdf = none_gdf.head(30)
            print(
                f"\n--- First {len(site_none_gdf)} none cells (site_geom not in scope) ---"
            )

        PROXY_KEYS = [
            "flood_score_mean",
            "risk_score_mean",
            "floodplain_adj_pct_mean",
            "depression_pct_mean",
            "merit_hand_mean",
            "imperv_pct_mean",
            "dw_built_pct_mean",
            "surface_water_occurrence_mean",
            "surface_water_seasonality_mean",
            "rx1day_2024_mean",
        ]

        def _riverine_strength_est(dist_river_m, riverine_flag):
            """Mirror nbs_rules.flood_mechanism_strengths riverine term from export fields."""
            if not riverine_flag:
                return 0.0
            if dist_river_m is None or (isinstance(dist_river_m, float) and pd.isna(dist_river_m)):
                return None
            dist = float(dist_river_m)
            if dist < 500:
                return round(max(0.0, 1.0 - dist / 500.0), 3)
            return 1.0  # riverine=True with dist ≥ 500 m → likely intersects waterway

        def _none_proxy_row_from_cell(cell):
            gs = cell.grid_stats
            strengths = cell.flood_mechanism.strengths if cell.flood_mechanism else {}
            dist = cell.water_stats.get("dist_nearest_m")
            row = {
                "cell_id": cell.cell_id,
                "flood_score": gs.get("flood_score_mean"),
                "dist_river_m": dist,
                "top_strength": max(strengths.values()) if strengths else None,
            }
            for key in PROXY_KEYS:
                row[key.replace("_mean", "")] = gs.get(key)
            for mech, val in strengths.items():
                row[f"s_{mech[:4]}"] = val
            return row

        def _none_proxy_row_from_gdf(row):
            dist = row.get("dist_river_m")
            riverine_flag = bool(row.get("riverine"))
            riverine_strength = _riverine_strength_est(dist, riverine_flag)
            out = {
                "cell_id": row["cell_id"],
                "flood_score": row.get("flood_score"),
                "risk_score": row.get("risk_score"),
                "dist_river_m": dist,
                "riverine": riverine_flag,
                "pluvial": row.get("pluvial"),
                "low_lying": row.get("low_lying"),
                "riverine_strength_est": riverine_strength,
                "top_strength_est": riverine_strength,
                "below_min_strength": (
                    riverine_strength < MIN_STRENGTH
                    if riverine_strength is not None
                    else None
                ),
            }
            return out

        if len(site_none_gdf):
            print("\n--- Per-cell proxies (none, bairro subset) ---")
            if poa_cells_by_id:
                proxy_rows = [
                    _none_proxy_row_from_cell(poa_cells_by_id[cid])
                    for cid in site_none_gdf["cell_id"]
                    if cid in poa_cells_by_id
                ]
            else:
                proxy_rows = [
                    _none_proxy_row_from_gdf(row)
                    for _, row in site_none_gdf.iterrows()
                ]
            display(pd.DataFrame(proxy_rows))

            print("\n--- Thresholds (nbs_rules.flood_mechanism_strengths) ---")
            for line in [
                f"dominant type assigned only if top strength ≥ {MIN_STRENGTH} (else → none)",
                "riverine: intersects waterway OR dist_river < 500 m",
                "pluvial: max(imperv, dw_built) ≥ 0.35 (or ≥ 0.25 + heavy rain from CHIRPS)",
                "low_lying: floodplain ≥ 0.5 OR depression ≥ 0.15 OR HAND ≤ 5 OR JRC occurrence ≥ 10",
                "drainage_constrained: imperv ≥ 0.3 + HAND ≤ 10 + riverine < 0.35",
            ]:
                print(f"  · {line}")

            if poa_cells_by_id:
                print("\n--- Rationale per none cell (bairro subset) ---")
                for cid in site_none_gdf["cell_id"]:
                    cell = poa_cells_by_id.get(cid)
                    if not cell:
                        continue
                    mech = cell.flood_mechanism
                    print(
                        f"\n{cell.cell_id}: flood_score={cell.grid_stats.get('flood_score_mean')} "
                        f"dist_river={cell.water_stats.get('dist_nearest_m')} "
                        f"strengths={mech.strengths if mech else {}}"
                    )
                    if mech:
                        for note in mech.rationale:
                            print(f"  · {note}")
            else:
                print(
                    "\n--- Rationale / full strengths skipped (GeoJSON export only) ---"
                )
                print(
                    "  riverine_strength_est = 1 − dist_river_m/500 (if riverine and dist < 500 m), "
                    "else 1.0 if riverine with dist ≥ 500 m (intersects), else 0."
                )
                print(
                    "  top_strength_est uses riverine only — pluvial/low_lying proxies are not in the GeoJSON."
                )
                print(
                    "  Re-run BUILD_POA_LAYER in this session for full proxy strengths and rationale."
                )


### Map — full POA `flood_mechanism_type` layer

Visualizes `output/flood_mechanism_type_poa_250m.tif` (preferred) or `.geojson` after `BUILD_POA_LAYER = True`, or from a prior export. The dashed outline is the current `SITE_NAME` bairro for context.

In [ ]:
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
from rasterio.plot import plotting_extent

if "MECHANISM_MIXED_COLOR" not in globals():
    MECHANISM_MIXED_COLOR = "#8e0152"

if "MECHANISM_COLORS" not in globals():
    MECHANISM_COLORS = {
        "none": "#e0e0e0",
        "riverine": "#2166ac",
        "pluvial": "#fdae61",
        "low_lying": "#abd9e9",
        "drainage_constrained": "#7b3294",
        "mixed": MECHANISM_MIXED_COLOR,
    }

POA_TIF = OUT_DIR / "flood_mechanism_type_poa_250m.tif"  # filled (observed + IDW)
POA_OBSERVED_TIF = OUT_DIR / "flood_mechanism_type_poa_250m_observed.tif"
POA_GEOJSON = OUT_DIR / "flood_mechanism_type_poa_250m.geojson"

TYPE_ORDER = ["none", "riverine", "pluvial", "low_lying", "drainage_constrained", "mixed"]
POA_CMAP = ListedColormap([MECHANISM_COLORS[t] for t in TYPE_ORDER])
POA_NORM = BoundaryNorm(np.arange(-0.5, len(TYPE_ORDER) + 0.5, 1), POA_CMAP.N)

fig, ax = plt.subplots(figsize=(11, 10))
source_label = None

if POA_TIF.exists():
    with rasterio.open(POA_TIF) as src:
        data = src.read(1, masked=True)
        # Export uses MECHANISM_RASTER_NODATA=255; code 0 = "none" is a valid class.
        if src.nodata != MECHANISM_RASTER_NODATA:
            data = np.ma.masked_equal(data, MECHANISM_RASTER_NODATA)
        ax.imshow(
            data,
            extent=plotting_extent(src),
            origin="upper",
            cmap=POA_CMAP,
            norm=POA_NORM,
            interpolation="nearest",
        )
    source_label = POA_TIF.name
elif POA_GEOJSON.exists():
    poa_gdf = gpd.read_file(POA_GEOJSON)
    poa_gdf = poa_gdf.copy()
    poa_gdf["color"] = poa_gdf["flood_mechanism_type"].map(MECHANISM_COLORS).fillna("#cccccc")
    poa_gdf.plot(ax=ax, color=poa_gdf["color"], edgecolor="none", linewidth=0, alpha=0.95)
    source_label = POA_GEOJSON.name
elif "poa_result" in globals():
    poa_gdf = cells_to_geodataframe(poa_result)
    poa_gdf = poa_gdf.copy()
    poa_gdf["color"] = poa_gdf["flood_mechanism_type"].map(MECHANISM_COLORS).fillna("#cccccc")
    poa_gdf.plot(ax=ax, color=poa_gdf["color"], edgecolor="none", linewidth=0, alpha=0.95)
    source_label = "in-memory poa_result"
else:
    print(
        "No POA layer found. Set BUILD_POA_LAYER=True in the cell above, "
        "or ensure output/flood_mechanism_type_poa_250m.tif|.geojson exists."
    )

if source_label:
    try:
        site_gdf.boundary.plot(
            ax=ax,
            color="black",
            linewidth=2,
            linestyle="--",
            label=SITE_NAME,
        )
    except NameError:
        pass

    ax.set_title(
        f"Dominant flood mechanism type — Porto Alegre (250 m, filled)\nsource: {source_label}"
    )
    ax.set_xlabel("lon")
    ax.set_ylabel("lat")
    legend_handles = [
        Patch(facecolor=MECHANISM_COLORS[t], edgecolor="white", label=t) for t in TYPE_ORDER
    ]
    ax.legend(handles=legend_handles, loc="lower left", fontsize=8, title="flood_mechanism_type")
    plt.tight_layout()
    plt.show()

    if POA_GEOJSON.exists():
        summary = json.loads(POA_GEOJSON.read_text()).get("properties", {}).get(
            "mechanism_summary", {}
        )
        counts = summary.get("dominant_mechanism_type_counts")
        if counts:
            print("POA dominant mechanism counts:")
            display(pd.Series(counts).sort_values(ascending=False).to_frame("cells"))

### Publish POA layer — COG + map tiles

Converts the **filled** `flood_mechanism_type_poa_250m.tif` (observed + IDW gap-fill) to **EPSG:3857 COG**, builds **visual** and **value** XYZ tiles (`z=8–15`), and writes the GDAL **colors** file. Requires GDAL CLI (`gdalwarp`, `gdal_translate`, `gdaldem`, `gdal_calc.py`, `gdal2tiles.py`). Run after `BUILD_POA_LAYER = True`.

In [ ]:
import os
import shutil
import subprocess

PUBLISH_POA_COG_TILES = True  # flip True after BUILD_POA_LAYER produces the .tif
UPLOAD_POA_TO_S3 = True  # requires AWS CLI (aws configure)

POA_LAYER_SLUG = "flood_mechanism_type_poa_250m"
IN_TIF = OUT_DIR / f"{POA_LAYER_SLUG}.tif"
POA_PUBLISH_DIR = OUT_DIR / POA_LAYER_SLUG
COLORS_TXT = POA_PUBLISH_DIR / f"{POA_LAYER_SLUG}_colors.txt"
WARPED_TIF = POA_PUBLISH_DIR / f"{POA_LAYER_SLUG}_3857.tif"
COG_TIF = POA_PUBLISH_DIR / f"{POA_LAYER_SLUG}_cog.tif"
COLORIZED_TIF = POA_PUBLISH_DIR / f"{POA_LAYER_SLUG}_colorized.tif"
VALUE_RGB_TIF = POA_PUBLISH_DIR / f"{POA_LAYER_SLUG}_value_encoded_rgb.tif"
VISUAL_TILES_DIR = POA_PUBLISH_DIR / "tiles_visual"
VALUE_TILES_DIR = POA_PUBLISH_DIR / "tiles_values"
VALUE_DECODE_TXT = POA_PUBLISH_DIR / f"{POA_LAYER_SLUG}_value_tiles_decode.txt"

_GDAL_CLI = (
    "gdalwarp",
    "gdal_translate",
    "gdaldem",
    "gdal_calc.py",
    "gdal2tiles.py",
)


def _hex_to_rgb(hex_color: str) -> tuple[int, int, int]:
    h = hex_color.lstrip("#")
    return tuple(int(h[i : i + 2], 16) for i in (0, 2, 4))


def _prepend_common_bin_to_path() -> None:
    for prefix in (Path("/opt/homebrew/bin"), Path("/usr/local/bin")):
        if prefix.is_dir():
            p = str(prefix)
            if p not in os.environ.get("PATH", "").split(":"):
                os.environ["PATH"] = f"{p}:{os.environ.get('PATH', '')}"


def _require_gdal_cli() -> None:
    _prepend_common_bin_to_path()
    missing = [cmd for cmd in _GDAL_CLI if not shutil.which(cmd)]
    if missing:
        raise RuntimeError(
            f"Missing GDAL CLI tools: {missing}. "
            "Install GDAL (e.g. brew install gdal) and restart the kernel, "
            "or ensure /opt/homebrew/bin is on PATH."
        )


if not PUBLISH_POA_COG_TILES:
    print(
        "Skipping POA COG/tiles publish (PUBLISH_POA_COG_TILES=False). "
        "Set True after flood_mechanism_type_poa_250m.tif exists."
    )
elif not IN_TIF.exists():
    raise FileNotFoundError(f"Missing input raster: {IN_TIF}. Run BUILD_POA_LAYER=True first.")
else:
    _require_gdal_cli()
    POA_PUBLISH_DIR.mkdir(parents=True, exist_ok=True)

    color_lines = [
        "# Dominant flood mechanism type (ON-5990). GDAL color-relief for visual tiles.",
        "# Codes from FLOOD_MECHANISM_TYPE_CODES in nbs_rules.py",
        f"# Raster nodata = {MECHANISM_RASTER_NODATA} (outside screened grid); code 0 = none",
        "nv 0 0 0 0",
    ]
    for mech_type, code in sorted(FLOOD_MECHANISM_TYPE_CODES.items(), key=lambda kv: kv[1]):
        r, g, b = _hex_to_rgb(MECHANISM_COLORS[mech_type])
        color_lines.append(f"{code} {r} {g} {b}")
    COLORS_TXT.write_text("\n".join(color_lines) + "\n", encoding="utf-8")
    print("Wrote colors:", COLORS_TXT)

    subprocess.run(
        ["gdalwarp", "-t_srs", "EPSG:3857", "-r", "near", "-overwrite", str(IN_TIF), str(WARPED_TIF)],
        check=True,
    )
    subprocess.run(
        [
            "gdal_translate",
            str(WARPED_TIF),
            str(COG_TIF),
            "-of",
            "COG",
            "-ot",
            "Byte",
            "-co",
            "COMPRESS=DEFLATE",
            "-co",
            "RESAMPLING=NEAREST",
            "-co",
            "OVERVIEWS=AUTO",
        ],
        check=True,
    )
    print("Created COG:", COG_TIF)

    subprocess.run(
        ["gdaldem", "color-relief", "-nearest_color_entry", str(COG_TIF), str(COLORS_TXT), str(COLORIZED_TIF), "-alpha"],
        check=True,
    )
    print("Created colorized raster:", COLORIZED_TIF)

    VISUAL_TILES_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["gdal2tiles.py", "-r", "near", "-z", "8-15", "--xyz", "-w", "none", str(COLORIZED_TIF), str(VISUAL_TILES_DIR)],
        check=True,
    )
    print("Visual tiles:", VISUAL_TILES_DIR)

    base_expr = (
        "numpy.where(numpy.isnan(A), 0, "
        "numpy.rint(numpy.clip(A,0,16777214)).astype(numpy.int64) + 1)"
    )
    subprocess.run(
        [
            "gdal_calc.py",
            "-A",
            str(COG_TIF),
            "--calc",
            f"bitwise_and({base_expr},255)",
            "--calc",
            f"bitwise_and(right_shift({base_expr},8),255)",
            "--calc",
            f"bitwise_and(right_shift({base_expr},16),255)",
            "--type",
            "Byte",
            "--NoDataValue",
            "0",
            "--overwrite",
            "--outfile",
            str(VALUE_RGB_TIF),
        ],
        check=True,
    )

    VALUE_TILES_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["gdal2tiles.py", "-r", "near", "-z", "8-15", "--xyz", "-w", "none", str(VALUE_RGB_TIF), str(VALUE_TILES_DIR)],
        check=True,
    )
    print("Value tiles:", VALUE_TILES_DIR)

    VALUE_DECODE_TXT.write_text(
        "\n".join(
            [
                "Flood mechanism type POA value tiles",
                "",
                f"Source raster: {IN_TIF}",
                f"COG (EPSG:3857): {COG_TIF}",
                f"Visual tiles: {VISUAL_TILES_DIR}/{{z}}/{{x}}/{{y}}.png",
                f"Value tiles: {VALUE_TILES_DIR}/{{z}}/{{x}}/{{y}}.png",
                "",
                "Value tile encoding (Terrain RGB style):",
                "encoded = R + 256 * G + 65536 * B",
                "if encoded == 0: nodata",
                "else: flood_mechanism_code = encoded - 1",
                "",
                "Mechanism codes:",
                *[f"  {code}: {mech_type}" for mech_type, code in sorted(FLOOD_MECHANISM_TYPE_CODES.items(), key=lambda kv: kv[1])],
            ]
        )
        + "\n",
        encoding="utf-8",
    )
    print("Value decode notes:", VALUE_DECODE_TXT)

    if UPLOAD_POA_TO_S3:
        import poa_mechanism_publish as _poa_publish

        _poa_publish.upload_poa_mechanism_to_s3(
            "flood",
            OUT_DIR,
            publish_dir=POA_PUBLISH_DIR,
            geojson_path=OUT_DIR / f"{POA_LAYER_SLUG}.geojson",
        )
    else:
        print("Skipping S3 upload (UPLOAD_POA_TO_S3=False).")

---

### Try another site

Change `SITE_NAME` in Setup (used for both bairro and grid sections).

```bash
geospatial-data/floods/.venv/bin/python transformation/nbs_screening/scripts/run_e2e.py --hazard flood
```
